In [ ]:
#| export machine_learning.information_note_types
import os
from os import PathLike
from pathlib import Path
from typing import Callable, Literal, Optional
import warnings

from fastai.text.learner import TextLearner
import torch

from trouver.obsidian.file import MarkdownFile
from trouver.personal_vault.note_processing import process_standard_information_note
from trouver.obsidian.vault import VaultNote


In [ ]:
from fastai.learner import load_learner
import pathlib
from pathlib import WindowsPath
import platform
import shutil
import tempfile
from unittest import mock

from fastcore.test import *
from fastcore.test import all_equal
from torch import tensor

from trouver.helper.tests import _test_directory
from trouver.personal_vault.notes import notes_linked_in_note

from trouver.machine_learning.information_note_types import LABEL_TAGS

## Use the trained model to predict note types

After training the model (cf. `how_to.train_ml_model.fastai`), we can now predict the types of notes

In [ ]:
#| export machine_learning.information_note_types
def possible_text_type_labels(
        learn: TextLearner
        ) -> list[str]:
    """Return the possible labels outputted by `learn.predict`
    """
    return learn.dls.vocab.items[1]

In [ ]:
#| export machine_learning.information_note_types
def predict_text_types_with_one_learner(
        learner: TextLearner, # The ML models predicting note types.
        texts: list[str],
        remove_NO_TAG: bool = True, # If `True`, remove `NO_TAG`, which in theory is supposed to indicate that no types are predicted, but in practice can somehow be predicted along with actual types.
        include_probabilities: bool = False, # If `True`, then  
        ) -> list[list[str] | tuple[list[str], dict[str, float]]]: # Each list or tuple corresponds to each entry from `text` and contains the predicted types of the text. A `list[str]` consists of the predicted types/labels of the text and a `tuple[list[str], dict[str,float]]` contains the list of predicted types along with a dict of all possible types predictable by `learn` along with probabilities.
    """Predict the types of mathematical texts using an ML model."""
    predictions = []
    for text in texts:
        with learner.no_bar(), learner.no_logging():
            pred, _, probabilities = learner.predict(text)
        if remove_NO_TAG and 'NO_TAG' in pred:
            pred.remove('NO_TAG')
        if include_probabilities:
            predictions.append(
                (list(pred), 
                 _make_probability_dict(probabilities,
                                        possible_text_type_labels(learner))))
        else:
            predictions.append(list(pred))
    return predictions


def _make_probability_dict(
        probabilities: list[torch.Tensor],
        possible_labels: list[str]
        ) -> dict[str, float]:
    return {label: prob.item() for label, prob in zip(possible_labels, probabilities)}

We can predict types of short mathematical texts. Say that the information note type classification model, trained in `how_to.train_ml_model.fastai` is loaded, e.g. via `fastai`'s `load_learner` function:

```python
model = load_learner(<path_to_model>)
```

In [ ]:
#| notest
#| hide
if platform.system() == 'Windows':
    folder = WindowsPath(r'C:\Users\hyunj\Documents\Development\ml_data')
    temp = pathlib.PosixPath  # See https://stackoverflow.com/questions/57286486/i-cant-load-my-model-because-i-cant-put-a-posixpath
    pathlib.PosixPath = pathlib.WindowsPath  # This makes sure that the model can be loaded
    model = load_learner(folder / 'information_note_type' / 'information_note_type_classification_model.pkl', cpu=True)
    pathlib.PosixPath = temp
elif platform.system() == 'Linux':
    folder = Path('/home/hyunjong/Documents/Development/ml_data')
    model = load_learner(folder / 'information_note_type' / 'information_note_type_classification_model.pkl')

In [ ]:
#| notest
texts_to_predict = [
    r'',
    r'In this chapter, we introduce the notion of rings, some related notions, and many examples.',
    r'A ring is a set equipped with two binary operators $+$ and $\cdot$ such that...',
    r'Theorem. For every prime power $q$, there is, up to isomorphism, exactly one field with $q$ elements.\n\nProof. Let $q = p^k$ where $p$ is a prime. Let $F$ be a field with $q$ elements. Note that...',
    r'Remark. Note that $\mathbb{F}_q$ and $\mathbb{Z}/q\mathbb{Z}$ are different rings',
    r'As an example, take $\mathbb{F}_9$. It can be presented as $\mathbb{F}_3[x^2+1]$ as well as $\mathbb{F}_3[x^2+x+2]$.'
]
sample_outputs = predict_text_types_with_one_learner(
    model, texts_to_predict)

print(sample_outputs)

[['#_meta/TODO/delete', '#_meta/TODO/merge', '#_meta/TODO/split', '#_meta/concept', '#_meta/proof'], ['#_meta/TODO/split'], ['#_meta/TODO/split', '#_meta/definition'], ['#_meta/concept', '#_meta/proof'], ['#_meta/TODO/split', '#_meta/remark'], ['#_meta/example', '#_meta/narrative']]


In [ ]:
#| notest
possible_text_type_labels(model)
sample_outputs = predict_text_types_with_one_learner(model, texts_to_predict, include_probabilities=True)
print(sample_outputs[0][0])
print(sample_outputs[0][1])
print(sample_outputs[0][1])

['#_meta/TODO/delete', '#_meta/TODO/merge', '#_meta/TODO/split', '#_meta/concept', '#_meta/proof']
{'#_meta/TODO/delete': 1.0, '#_meta/TODO/merge': 0.8483226299285889, '#_meta/TODO/split': 0.9903709292411804, '#_meta/concept': 0.6789887547492981, '#_meta/conjecture': 8.332793656473658e-12, '#_meta/convention': 0.37105536460876465, '#_meta/definition': 4.3158637708984315e-05, '#_meta/example': 4.248961886332836e-06, '#_meta/exercise': 1.3349515029403847e-05, '#_meta/how_to': 6.355448567774147e-05, '#_meta/narrative': 7.353986575253657e-07, '#_meta/notation': 1.661644273553975e-05, '#_meta/proof': 0.9312138557434082, '#_meta/remark': 0.001952954800799489, 'NO_TAG': 4.7666364189069554e-09}
{'#_meta/TODO/delete': 1.0, '#_meta/TODO/merge': 0.8483226299285889, '#_meta/TODO/split': 0.9903709292411804, '#_meta/concept': 0.6789887547492981, '#_meta/conjecture': 8.332793656473658e-12, '#_meta/convention': 0.37105536460876465, '#_meta/definition': 4.3158637708984315e-05, '#_meta/example': 4.24896

In [ ]:
#| export machine_learning.information_note_types
def consolidate_single_text_predictions_by_sum_of_confidence(
        predictions_for_single_text: list[tuple[list[str], dict[str, float]]] # Each tuple corresponds to the predictions made by each model.
        ) -> list[str]: # The labels
    """
    Consolidate single text predictions by summing the "probabilities"
    predicted by the various models. If the sum of the probabilities that
    the label should be predicted is greater than the sum of the probabilities
    that the label should not be predicted, then the label is predicted.

    This is a sample input to the `consolidation` parameter of the
    `predict_note_types` function

    """
    all_keys = set.union(
        *[set(probs.keys()) for _, probs in predictions_for_single_text])
    # If tally is positive in the end, then the label is predicted
    # Otherwise, the label is not predicted.
    tally = {key: 0 for key in all_keys}
    for _, probs in predictions_for_single_text:
        for key in tally:
            if key in probs:
                tally[key] += probs[key] - (1- probs[key])
    return [key for key in tally if tally[key] > 0]

In [ ]:
#| export machine_learning.information_note_types
def predict_note_types(
        learners: TextLearner|list[TextLearner], # The ML models predicting note types.
        vault: PathLike, # The vault with the notes.
        notes: list[VaultNote], # The notes with texts to predict
        remove_NO_TAG: bool = True, # If `True`, remove `NO_TAG`, which in theory is supposed to indicate that no types are predicted, but in practice can somehow be predicted along with actual types.
        consolidation: Optional[Callable[[list[tuple[list[str], dict[str, float]]]], list[str]]] = consolidate_single_text_predictions_by_sum_of_confidence, # The method to consolidate between different predictions made by the possibly more-than-one model in `learners`.
        ) -> list[list[str]]: # Each `list[str]`` corresponds to an item in `notes` and contains the predicted note types for that note.
    """

    **Parameters**

    **Returns**


    """
    if not isinstance(learners, list):
        learners = [learners]
    markdown_files = [
        MarkdownFile.from_vault_note(note) for note in notes]
    raw_note_contents = [
        str(process_standard_information_note(mf, vault)) for mf in markdown_files]
    predictions_by_learners = [
        predict_text_types_with_one_learner(
            learner, raw_note_contents, remove_NO_TAG,
            include_probabilities=True)
        for learner in learners]

    predictions_by_texts = _transpose_list(predictions_by_learners)
    consolidated_predictions = [
        consolidation(predictions_by_text)
        for predictions_by_text in predictions_by_texts]
    if remove_NO_TAG:
        for preds in consolidated_predictions:
            if 'NO_TAG' in preds:
                preds.remove('NO_TAG')
    return consolidated_predictions




    # return predict_text_types(learners, raw_note_contents, remove_NO_TAG)

def _transpose_list(original_list: list[list]):
    return list(map(list, zip(*original_list)))


In [ ]:
#| notest
# TODO: tests
# predict_note_types

test_vault = _test_directory() / 'test_vault_6'
notes_to_predict = [
    VaultNote(test_vault, name='reference_with_tag_labels_Theorem 1'),
    VaultNote(test_vault, name='reference_with_tag_labels_Definition 1')
]
sample_outputs = predict_note_types(model, test_vault, notes_to_predict)

print(
    f'The following are the raw content of the notes without'
    f'metadata along with the model\'s predictions for their types:\n\n')
for note, prediction in zip(notes_to_predict, sample_outputs):
    print(process_standard_information_note(MarkdownFile.from_vault_note(note), test_vault) )
    print(prediction, '\n\n')

C:\Users\hyunj\Documents\Development\Python\trouver\trouver\helper\html.py:99: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  parsed_soup = BeautifulSoup(text, 'html.parser')


The following are the raw content of the notes withoutmetadata along with the model's predictions for their types:


Theorem 1. Let $R$ be a UFD. Then $R[x]$ is a UFD.

Proof. Let $f,g \in R[x]$ and suppose that $fg = 0$. Write $f = \sum_{i=0}^n a_i x^i$ and $g = \sum_{j=0}^m b_j x^j$ for some $a_i,b_j \in R$.

...

['#_meta/proof', '#_meta/concept'] 


A ring is a set with binary operators $+$ and $\cdot$ such that ...

['#_meta/TODO/split', '#_meta/definition'] 




C:\Users\hyunj\Documents\Development\Python\trouver\trouver\helper\html.py:99: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  parsed_soup = BeautifulSoup(text, 'html.parser')


In [ ]:
with (mock.patch('__main__.MarkdownFile.from_vault_note') as mock_markdownfile_from_vault_note,
          mock.patch('__main__.process_standard_information_note') as mock_process_standard_information_note,
          mock.patch('__main__.load_learner') as mock_load_learner,
          mock.patch('__main__.possible_text_type_labels') as mock_possible_text_type_labels):
      mock_path, mock_vault = None, None
      mock_notes = [None, None, None, None, None]
      mock_learner = load_learner(mock_path)
      mock_possible_text_type_labels.return_value = ['#_meta/TODO/delete', '#_meta/TODO/merge', '#_meta/TODO/split', '#_meta/concept', '#_meta/conjecture', '#_meta/convention', '#_meta/definition', '#_meta/example', '#_meta/exercise', '#_meta/how_to', '#_meta/narrative', '#_meta/notation', '#_meta/proof', '#_meta/remark', 'NO_TAG']
      mock_learner.predict.side_effect = [
          (['#_meta/TODO/delete','#_meta/TODO/split','#_meta/concept','#_meta/convention'],
            tensor([ True, False,  True,  True, False,  True, False, False, False, False,
                    False, False, False, False, False]),
            tensor([1.0000e+00, 1.0748e-02, 8.5110e-01, 8.2098e-01, 3.0918e-09, 8.9845e-01,
                    6.3798e-03, 1.1441e-06, 2.3794e-07, 1.7402e-02, 3.2704e-06, 1.6747e-03,
                    2.1008e-01, 3.8969e-04, 3.7873e-07])),
          (['#_meta/narrative'],
            tensor([False, False, False, False, False, False, False, False, False, False,
                      True, False, False, False, False]),
            tensor([8.4556e-06, 3.2360e-06, 2.0235e-01, 1.1844e-03, 5.1291e-08, 2.9886e-07,
                    4.8174e-04, 8.9895e-06, 1.3379e-09, 2.8261e-11, 9.9701e-01, 5.8664e-04,
                    2.6031e-03, 1.4012e-02, 1.3595e-03])),
          (['#_meta/TODO/delete', 'NO_TAG'], tensor([ True, False, False, False, False, False, False, False, False, False,
            False, False, False,  True]), tensor([6.1455e-01, 1.6902e-01, 1.4254e-02, 5.1358e-02, 4.8857e-05, 3.0853e-04,
            6.0457e-03, 1.2064e-04, 8.5651e-02, 1.3941e-02, 5.2413e-04, 1.3709e-01,
            9.7726e-06, 0.00, 9.7614e-01])),
          (['#_meta/concept', '#_meta/proof'], tensor([False, False, False,  True, False, False, False, False, False, False,
            False,  True, False, False]), tensor([4.0871e-03, 3.6683e-04, 1.6594e-01, 9.7876e-01, 0.00, 6.0281e-05, 8.7817e-06,
            1.9275e-02, 1.5589e-03, 4.5301e-03, 7.8989e-03, 1.2528e-02, 9.2800e-01, 
            9.4636e-04, 1.4658e-02])),
          (['NO_TAG'], tensor([ True, False, False, False, False, False, False, False, False, False,
            False, False, False,  True]), tensor([6.1455e-02, 1.6902e-01, 1.4254e-02, 5.1358e-02, 4.8857e-05, 3.0853e-04,
            6.0457e-03, 1.2064e-04, 8.5651e-02, 1.3941e-02, 5.2413e-04, 1.3709e-01,
            9.7726e-06, 0.00, 9.7614e-01])),
      ]
      prediction = predict_note_types(mock_learner, mock_vault, mock_notes)
      correct_value = [['#_meta/TODO/delete', '#_meta/TODO/split', '#_meta/convention', '#_meta/concept'],
         ['#_meta/narrative'],
         ['#_meta/TODO/delete'],
         ['#_meta/proof', '#_meta/concept'],
         []]
      test_eq(len(prediction), len(correct_value))
      for first, second in zip(prediction, correct_value):
          try:
              test(first, second, all_equal)
          except AssertionError:
               test_shuffled(first, second)

      mock_notes = [None, None]
      mock_learner.predict.side_effect = [
          (['#_meta/TODO/delete', 'NO_TAG'], tensor([ True, False, False, False, False, False, False, False, False, False,
            False, False, False,  True]), tensor([6.1455e-01, 1.6902e-01, 1.4254e-02, 5.1358e-02, 4.8857e-05, 3.0853e-04,
            6.0457e-03, 1.2064e-04, 8.5651e-02, 1.3941e-02, 5.2413e-04, 1.3709e-01,
            9.7726e-06, 0.00, 9.7614e-01])),
          (['NO_TAG'], tensor([ True, False, False, False, False, False, False, False, False, False,
            False, False, False,  True]), tensor([6.1455e-02, 1.6902e-01, 1.4254e-02, 5.1358e-02, 4.8857e-05, 3.0853e-04,
            6.0457e-03, 1.2064e-04, 8.5651e-02, 1.3941e-02, 5.2413e-04, 1.3709e-01,
            9.7726e-06, 0.00, 9.7614e-01])),
      ]

      prediction = predict_note_types(mock_learner, mock_vault, mock_notes, remove_NO_TAG=False)
      correct_value = [['NO_TAG', '#_meta/TODO/delete'],
         ['NO_TAG']]
      test_eq(len(prediction), len(correct_value))
      for first, second in zip(prediction, correct_value):
          try:
              test(first, second, all_equal)
          except AssertionError:
               test_shuffled(first, second)

In [ ]:
#| export machine_learning.information_note_types
def automatically_add_note_type_tags(
        learners: TextLearner|list[TextLearner], # The ML model(s) predicting note types.
        vault: PathLike, # The vault with the notes
        notes: list[VaultNote],
        add_auto_label: bool = True, # If `True`, adds `"_auto"` to the front of the note type tag to indicate that the tags were added via this automated script.
        overwrite: Optional[Literal['w', 'ws', 'ww', 'a', None]] = None # Either `'w'`, `'ws'`, `'ww'`, `'a'`, or `None`. If `'w'` or `'ws'`, then overwrite any already-existing note type tags (from LABEL_TAGS), whether or not these tags are `_auto` tags, with the predicted tags. IF `'ww'`, then overwrite only the `_auto` tags among the already-existing note type tags with the predicted tags. If `'a'`, then preserve already-existing note type tags and just append the newly predicted ones; in the case that `learn` predicts the note type whose tag is already in the note, a new tag of that type is not added, even if `add_auto_label=True`. If `None`, then do not make modifications to each note if any note type tags already exist in the note; if the predicted note types are different from the already existing note types, then raise a warning.
        ) -> None:
    """
    Predict note types and add the predicted types as
    frontmatter YAML tags in the notes.

    Non-`_auto`-labeled tags take precedent over `auto`-labeled tags,
    unless `overwrite='w'`.
    
    **Raises**

    - Warning:
        - If `overwrite=None`, a note already has some note type tags,
        and `learn` predicts different note types as those in the note.
    
    """
    if not isinstance(learners, list):
        learners = [learners]
    if overwrite not in ['w', 'ws', 'ww', 'a'] and overwrite is not None:
        raise ValueError(
            f"`overwrite` was expected to be 'w', 'ws,', 'ww', 'a', or None," 
            f" but was {overwrite}")
    predictions = predict_note_types(learners, vault, notes)
    # remove hashtags
    predictions = [[tag[1:] if tag.startswith('#') else tag for tag in tags]
                   for tags in predictions]
    # Add `_auto/`
    all_label_tags = [*LABEL_TAGS]
    all_label_tags.extend([f'_auto/{tag}' for tag in LABEL_TAGS])
    for note, prediction in zip(notes, predictions):
        _change_label_tags_for_single_note(
            note, prediction, overwrite, add_auto_label,
            all_label_tags)


def _change_label_tags_for_single_note(
        note: VaultNote, prediction: list[str], overwrite: Optional[str],
        add_auto_label: bool, all_label_tags: list[str]):
    mf = MarkdownFile.from_vault_note(note)

    if add_auto_label:
        tags_to_add = _auto_prediction(prediction)
    else:
        tags_to_add = prediction

    if overwrite in ['w', 'ws']:
        mf.remove_tags(all_label_tags)
        mf.add_tags(tags_to_add, skip_repeated_auto=True)
    elif overwrite == 'ww':
        mf.remove_tags(
            [tag for tag in all_label_tags if tag.startswith('_auto/')])
        mf.add_tags(tags_to_add, skip_repeated_auto=True)
    elif overwrite == 'a':
        for tag in prediction:
            _append_single_predicted_tag(mf, tag, add_auto_label)
    else:  # overwrite=None
        if not _has_any_label_tags(mf):
            mf.add_tags(tags_to_add, skip_repeated_auto=True)
        elif not _has_exactly_predicted_tags(mf, prediction):
            warnings.warn(
                "The note type labeling tags in the note are different "
                "from the predicted note types. "
                f"The note type tags in the note have NOT been modified:"
                f"\n\nNote name: {note.name}"
                f"\n\nPredicted types: {prediction}", UserWarning)        
    mf.write(note)


def _auto_prediction(prediction: list[str]):
    return [f'_auto/{tag}' for tag in prediction]


def _append_single_predicted_tag(
        mf, tag, add_auto_label):
    if tag in mf.metadata()['tags']:
        return
    elif f'_auto/{tag}' in mf.metadata()['tags'] and add_auto_label:
        return
    elif f'_auto/{tag}' in mf.metadata()['tags'] and not add_auto_label:
        mf.remove_tags([f'_auto/{tag}'])
        mf.add_tags([tag])
    else:
        mf.add_tags([f'_auto/{tag}'] if add_auto_label else [tag])
    

def _has_exactly_predicted_tags(
        mf, prediction: list[str]) -> bool:
    """Return `True` if the MarkdownFile already has the predicted tags
    (or the corresponding `_auto` tags)"""
    for tag in LABEL_TAGS:
        if (mf.has_tag(tag) or mf.has_tag(f'_auto/{tag}')) and tag in prediction:
            continue
        else:
            return False
    return True


def _has_any_label_tags(
        mf) -> bool:
    """Return `True` if the MarkdownFile has any label tags (or correspnoding `_auto` tags)"""
    for tag in LABEL_TAGS:
        if mf.has_tag(tag) or mf.has_tag(f'_auto/{tag}'):
            continue
        else:
            return False
    return True

In the below examples, we use `mock.patch` to test adding note types without testing the ML model itself. In particular, we pretend as though the ML model returns certain predictions (technically, we pretend as though `predict_note_types` return certain values) to construct these examples.

The following example demonstrates a basic use case of adding predicted note type tags to notes without any note type tags:

In [ ]:
# Here we just test adding a note type without testing the ML model itself.
with (tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir,
      mock.patch('__main__.predict_note_types') as mock_predict_note_types):
    temp_vault = Path(temp_dir) / 'test_vault_6'
    shutil.copytree(_test_directory() / 'test_vault_6', temp_vault)

    # Example where `add_auto_label` is `True`
    mock_predict_note_types.return_value = [['#_meta/definition'], ['#_meta/concept', '#_meta/proof']]
    vn1 = VaultNote(temp_vault, name='reference_without_tag_labels_Definition 1')
    vn2 = VaultNote(temp_vault, name='reference_without_tag_labels_Theorem 1')
    notes = [vn1, vn2]
    mock_learn = None
    automatically_add_note_type_tags(mock_learn, temp_vault, notes)

    # Note that _auto/_meta/definition has been added
    print(vn1.text())
    assert (MarkdownFile.from_vault_note(vn1).has_tag('_auto/_meta/definition'))
    assert (MarkdownFile.from_vault_note(vn2).has_tag('_auto/_meta/concept'))
    assert (MarkdownFile.from_vault_note(vn2).has_tag('_auto/_meta/proof'))


    # Examle where `add_auto_label` is `False`
    mock_predict_note_types.return_value = [['#_meta/definition', '#_meta/notation']]
    notes = [VaultNote(temp_vault, name='reference_without_tag_labels_Definition 2')]
    automatically_add_note_type_tags(mock_learn, temp_vault, notes, add_auto_label=False)
    assert (MarkdownFile.from_vault_note(notes[0]).has_tag('_meta/definition'))
    assert (MarkdownFile.from_vault_note(notes[0]).has_tag('_meta/notation'))
    

---
cssclass: clean-embeds
aliases: []
tags: [_auto/_meta/definition, _meta/literature_note]
---
# Ring[^1]

A **ring** is a set with binary operators $+$ and $\cdot$ such that ...

# See Also

# Meta
## References

## Citations and Footnotes
[^1]: Kim, Definition 1


In the following example, `overwrite` is set to `'w'` (or `'ws'`), so all preexisting note type tags are removed before the predicted ones are added:

In [ ]:
with (tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir,
      mock.patch('__main__.predict_note_types') as mock_predict_note_types):
    temp_vault = Path(temp_dir) / 'test_vault_6'
    shutil.copytree(_test_directory() / 'test_vault_6', temp_vault)

    mock_predict_note_types.return_value = [['#_meta/definition']]
    vn1 = VaultNote(temp_vault, name='reference_with_tag_labels_Definition 1')
    notes = [vn1]
    mock_learn = None
    automatically_add_note_type_tags(mock_learn, temp_vault, notes, overwrite='w')

    # Note that _meta/definition has been removed and _auto/_meta/definition has been added.
    print(vn1.text())
    assert MarkdownFile.from_vault_note(vn1).has_tag('_auto/_meta/definition')
    assert not MarkdownFile.from_vault_note(vn1).has_tag('_meta/definition')

---
cssclass: clean-embeds
aliases: []
tags: [_auto/_meta/definition, _meta/literature_note]
---
# Ring[^1]

A **ring** is a set with binary operators $+$ and $\cdot$ such that ...

# See Also

# Meta
## References

## Citations and Footnotes
[^1]: Kim, Definition 1


In the following example, `overwrite` is set to `'ww'`, so only the `_auto` note type tags are removed and the predicted ones are added:

In [ ]:
# TODO: change example
with (tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir,
      mock.patch('__main__.predict_note_types') as mock_predict_note_types):
    temp_vault = Path(temp_dir) / 'test_vault_6'
    shutil.copytree(_test_directory() / 'test_vault_6', temp_vault)

    mock_predict_note_types.return_value = [['#_meta/definition']]
    vn1 = VaultNote(temp_vault, name='reference_with_tag_labels_Definition 1')
    notes = [vn1]
    mock_learn = None
    automatically_add_note_type_tags(mock_learn, temp_vault, notes, overwrite='w')

    # Note that _meta/definition has been removed and _auto/_meta/definition has been added.
    print(vn1.text())
    assert MarkdownFile.from_vault_note(vn1).has_tag('_auto/_meta/definition')
    assert not MarkdownFile.from_vault_note(vn1).has_tag('_meta/definition')

---
cssclass: clean-embeds
aliases: []
tags: [_auto/_meta/definition, _meta/literature_note]
---
# Ring[^1]

A **ring** is a set with binary operators $+$ and $\cdot$ such that ...

# See Also

# Meta
## References

## Citations and Footnotes
[^1]: Kim, Definition 1


In the following example, `overwrite` is set to `'a'`, so only newly predicted note type tags are added

In [ ]:
with (tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir,
      mock.patch('__main__.predict_note_types') as mock_predict_note_types):
    temp_vault = Path(temp_dir) / 'test_vault_6'
    shutil.copytree(_test_directory() / 'test_vault_6', temp_vault)

    mock_predict_note_types.return_value = [['#_meta/definition', '#_meta/notation', '#_meta/concept', '#_meta/proof']]
    vn1 = VaultNote(temp_vault, name='reference_with_tag_labels_Theorem 2')
    notes = [vn1]
    mock_learn = None
    automatically_add_note_type_tags(mock_learn, temp_vault, notes, overwrite='a')

    # Example with `add_auto_label=True`
    # Note that _auto/_meta/notation has been added, but _meta/definition, _meta/concept,
    # and #_auto/_meta/proof remain unchanged. Moreover, _auto/_meta/definition and _auto/_meta/cocnept are
    # NOT added.
    print(vn1.text())
    assert MarkdownFile.from_vault_note(vn1).has_tag('_auto/_meta/notation')
    assert MarkdownFile.from_vault_note(vn1).has_tag('_meta/definition')
    assert MarkdownFile.from_vault_note(vn1).has_tag('_meta/concept')
    assert MarkdownFile.from_vault_note(vn1).has_tag('_auto/_meta/proof')
    assert not MarkdownFile.from_vault_note(vn1).has_tag('_auto/_meta/definition')
    assert not MarkdownFile.from_vault_note(vn1).has_tag('_auto/_meta/concept')


with (tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir,
      mock.patch('__main__.predict_note_types') as mock_predict_note_types):
    temp_vault = Path(temp_dir) / 'test_vault_6'
    shutil.copytree(_test_directory() / 'test_vault_6', temp_vault)

    mock_predict_note_types.return_value = [['#_meta/definition', '#_meta/notation', '#_meta/concept', '#_meta/proof']]
    vn1 = VaultNote(temp_vault, name='reference_with_tag_labels_Theorem 2')
    notes = [vn1]
    mock_learn = None
    automatically_add_note_type_tags(mock_learn, temp_vault, notes, overwrite='a', add_auto_label=False)

    # Example with `add_auto_label=False`
    # Note that _meta/notation has been added, and _auto/_meta/proof is replaced
    # with _meta/proof, but _meta/definition and _meta/concept remain unchanged.
    print(vn1.text())
    assert MarkdownFile.from_vault_note(vn1).has_tag('_meta/notation')
    assert MarkdownFile.from_vault_note(vn1).has_tag('_meta/proof')
    assert not MarkdownFile.from_vault_note(vn1).has_tag('_auto/_meta/proof')
    assert MarkdownFile.from_vault_note(vn1).has_tag('_meta/definition')
    assert MarkdownFile.from_vault_note(vn1).has_tag('_meta/concept')

---
cssclass: clean-embeds
aliases: []
tags: [_meta/literature_note, _auto/_meta/proof, _auto/_meta/notation, _meta/concept, _meta/definition]
---
%%Note that this note introduces a notation and hence actually ought to be labeled with the tag _meta/notation as well; but for the sake of example, the job of adding the _meta/notation tag will be left to the `automatically_add_note_type_tags` function.%%

# The polynomial ring of a UFD is a UFD[^1]
Let $q$ be the power of a prime number. Up to isomorphism, there is a unique field with $q$ elements. This field is denoted **$\mathbb{F}_q$** and is called the **finite field of $q$ elements**.

Proof. Say that $q = p^k$ and let $F$ be a field with $q$ elements. First note that $F$ has a subfield "generated by $1$", i.e. the elements $0,1,\ldots,p-1$ form a subfield of $F$.

# See Also

# Meta
## References

## Citations and Footnotes
[^1]: Kim, Theorem 2
---
cssclass: clean-embeds
aliases: []
tags: [_meta/literature_note, _meta/notation, _meta

In the following example, `overwrite` is set to `None`. The notes are not modified, but if the note type tags in a note do not match the predicted ones, then a warning is raised

In [ ]:
# TODO: add example

## Convert `_auto/` tags to regular tags

After checking that the automatically predicted tags are correct, we can convert them to regular tags.

In [ ]:
#| export machine_learning.information_note_types
def convert_auto_tags_to_regular_tags_in_notes(
        notes: list[VaultNote], 
        exclude: list[str] = ['links_added', 'notations_added'] # The tags whose `_auto/` tags should not be converted. The str should not start with `'#'` and should not start with `'_auto/'`.
        ) -> None:
    """Convert the auto tags into regular tags for the notes.
    """
    for note in notes:
        mf = MarkdownFile.from_vault_note(note)
        mf.replace_auto_tags_with_regular_tags(exclude)
        mf.write(note)

In [ ]:
with (tempfile.TemporaryDirectory(prefix='temp_dir', dir=os.getcwd()) as temp_dir):
    temp_vault = Path(temp_dir) / 'test_vault_6'
    shutil.copytree(_test_directory() / 'test_vault_6', temp_vault)

    vn = VaultNote(temp_vault, name='reference_with_tag_labels_Theorem 2')
    convert_auto_tags_to_regular_tags_in_notes([vn])
    print(vn.text())
    mf = MarkdownFile.from_vault_note(vn)
    assert mf.has_tag('_meta/proof')
    assert not mf.has_tag('_auto/_meta/proof')

---
cssclass: clean-embeds
aliases: []
tags: [_meta/literature_note, _meta/concept, _meta/proof, _meta/definition]
---
%%Note that this note introduces a notation and hence actually ought to be labeled with the tag _meta/notation as well; but for the sake of example, the job of adding the _meta/notation tag will be left to the `automatically_add_note_type_tags` function.%%

# The polynomial ring of a UFD is a UFD[^1]
Let $q$ be the power of a prime number. Up to isomorphism, there is a unique field with $q$ elements. This field is denoted **$\mathbb{F}_q$** and is called the **finite field of $q$ elements**.

Proof. Say that $q = p^k$ and let $F$ be a field with $q$ elements. First note that $F$ has a subfield "generated by $1$", i.e. the elements $0,1,\ldots,p-1$ form a subfield of $F$.

# See Also

# Meta
## References

## Citations and Footnotes
[^1]: Kim, Theorem 2
